# Preparation of metrics for participant-day analyses

Johannes Zauner

## Preface

This document collects the raw light exposure data for all three wearing positions through the `melidosData` package, preprocesses it to

1.  An unaggregated set of light exposure data, both for the eye-level and chest-level wearing position where

- Values \> 100 000 lx are removed (set to `NA`)
- non-wear periods, that are not also sleep periods, are removed
- remove observations where not all three wearing positions are available

1.  A further processed set (from 1.) where

- hours with less than 50% data availability are removed
- days with less than 80% data availability (after the previous step) are removed

With the cleaned dataset, metrics per participant and day are calculated (with explicit exceptions). The resulting dataset is the basis for downstream inferential analyses.

## Setup

In [ ]:
library(tidyverse)


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

here() starts at /Users/zauner/Documents/Arbeit/12-TUM/Repos_Misc/WearingPositionError_2026


Attaching package: 'rlang'

The following objects are masked from 'package:purrr':

    flatten, flatten_chr, flatten_dbl, flatten_int, flatten_lgl,
    flatten_raw, invoke, splice


Attaching package: 'cowplot'

The following object is masked from 'package:gt':

    as_gtable

The following object is masked from 'package:lubridate':

    stamp

## Importing data

### Light

In the first step, we import the unaggregated light eposure data for the glasses, chest, and wrist. These will be loaded from the [GitHub project page](https://github.com/MeLiDosProject). These data have been imported already, trimmed by trial dates, checked for irregular data and gaps. All of them run on a `10 second` interval.

In [ ]:
light_glasses <- load_data("light_glasses")


These datasets are comparatively large (~50MB per site). Download may take a while.

loading modality: light_glasses ■■■■                              11% |  ETA:  …

loading modality: light_glasses ■■■■■■■■                          22% |  ETA:  …

loading modality: light_glasses ■■■■■■■■■■■                       33% |  ETA:  …

loading modality: light_glasses ■■■■■■■■■■■■■■                    44% |  ETA: 4…

...

loading modality: light_wrist ■■■■■■■■■■■■                      38% |  ETA: 47s

loading modality: light_wrist ■■■■■■■■■■■■■■■■                  50% |  ETA: 34s

loading modality: light_wrist ■■■■■■■■■■■■■■■■■■■■              62% |  ETA: 29s

loading modality: light_wrist ■■■■■■■■■■■■■■■■■■■■■■■           75% |  ETA: 18s

loading modality: light_wrist ■■■■■■■■■■■■■■■■■■■■■■■■■■■       88% |  ETA:  8s

In [ ]:
light_glasses |> flatten_data() |> group_by(Id) |> n_groups()


[1] 143

[1] 157

[1] 153

[1] 1136

[1] 1249

[1] 1223

### Sleep

The sleep data comes from a morning sleep diary. They contain both sleep and wake times.

In [ ]:
sleepdiary <- load_data("sleepdiaries")


loading modality: sleepdiary ■■■■■■■■■■■■■■■■■■■■■■■■■■■■      89% |  ETA:  0s

### Wear log

The non-wear data comes from an app-based wear log that participants filled in whenever they removed or put on the device(s).

In [ ]:
wearlog <- load_data("wearlog")


loading modality: wearlog ■■■■■■■■■■■■■■■■■■■■■■■■          78% |  ETA:  1s

### Combine wearing positions

- Only a selection of variables will be kept, namely `MEDI` (melanopic EDI) and `LIGHT` (photopic illuminance).

- Other contextual variables that are kept are `Id`, `Datetime`, and `position`.

- We further remove the `MPI` site from the glasses dataset, as no chest or wrist[1] data were collected.

- Then we will combine the observations of the wearing positions.

- Lastly, we will calculate the photoperiod information for each site.

[1] Wrist data were collected but with a different device (`ActTrust` instead of `ActLumus`). Thus, it will not be considered here.

In [ ]:
light_glasses$MPI <- NULL


In [ ]:
light <-
  imap(
    light_glasses,
    \(data, idx) data |>
      select(Id, Datetime, MEDI, LIGHT) |> 
      data2reference(light_chest[[idx]], Reference.column = MEDI_chest) |> 
      data2reference(light_chest[[idx]],
                     Data.column = LIGHT,
                     Reference.column = LIGHT_chest) |> 
      data2reference(light_wrist[[idx]], Reference.column = MEDI_wrist) |>
      data2reference(light_wrist[[idx]],
                     Data.column = LIGHT,
                     Reference.column = LIGHT_wrist) |>
      add_photoperiod(melidos_coordinates[[idx]]) |> 
      rename(MEDI_glasses = MEDI, LIGHT_glasses = LIGHT)
  )


Warning in interval2state(., State.interval.dataset = Reference.data, State.colname = {: The time differences between consecutive time points in the reference dataset are larger than in the dataset. This means multiple reference data connect to one dataset datum - only the last one prior to each datum will be used. Please use an aggregate function on the reference dataset to resolve this warning. 
The following output shows what grouping is problematic and what 95% of time intervals in the Dataset compared to the Reference data is.

  Id           Dataset.Interval Reference.Interval
1 FUSPCEU_S014               60                 10
Warning in interval2state(., State.interval.dataset = Reference.data, State.colname = {: The time differences between consecutive time points in the reference dataset are larger than in the dataset. This means multiple reference data connect to one dataset datum - only the last one prior to each datum will be used. Please use an aggregate function on the re

> Note: the displayed warnings refer to the fact that participant `FUSPCEU_S014` had a measurement interval of 60 seconds on the glasses, compared to the 10 seconds of everyone else (and both the other wearing positions). Thus, six measurement values from the chest and wrist positions map to one from the glasses. This is not deemed problematic, as it is only one participant in one site. The last of the six respective chest and wrist measurements will be used for comparison.

### Remove instances with less than all wearing positions

In [ ]:
light <- 
  light |> 
  map(
    \(data) data |> 
              mutate(
                across(MEDI_glasses:LIGHT_wrist,
                \(x) {
                ifelse(is.na(MEDI_glasses) | is.na(MEDI_chest) | is.na(MEDI_wrist),
                       NA, x)
                }
              )
              )
  )


### Combine light exposure data with log and diary data

For the sleepdiary, a selection of sleep and wake times will be added. As the variable `sleepprep` (preparation for sleep) is the more reasonable time indicator for when the device positions are set for the night, this variable will be used instead of the calculated time of `sleep` (`sleepprep` + `sleepdelay`). For the wearlog, the start and end times of a removal, as well as the type (`state`) of removal will be used.

#### Preparation

In [ ]:
sleepdiary_adj <-
  sleepdiary |>
  map(\(x) x |>
        select(Id, sleepprep, wake) |>
        group_by(Id) |>
        pivot_longer(-Id, names_to = "sleep", values_to = "Datetime") |>
        sc2interval(Statechange.colname = sleep, starting.state = "wake") |>
        sleep_int2Brown(sleep.state = "sleepprep", Brown.day = "wake", 
                        Brown.evening = "pre-sleep", Brown.night = "sleep") |>
        mutate(sleep = case_when(is.na(sleep) & State.Brown == "pre-sleep" ~ "wake",
                                 .default = sleep))
  )


Adding missing grouping variables: `Id`
Adding missing grouping variables: `Id`
Adding missing grouping variables: `Id`
Adding missing grouping variables: `Id`
Adding missing grouping variables: `Id`
Adding missing grouping variables: `Id`
Adding missing grouping variables: `Id`
Adding missing grouping variables: `Id`
Adding missing grouping variables: `Id`

#### Combination

In [ ]:
light <- 
light |> 
  imap(\(x, idx) x |>
    add_states(sleepdiary_adj[[idx]], start = Interval, end = Interval) |>
    add_states(wearlog_adj[[idx]])
  )


In [ ]:
light |> 
  structure(class = "melidos_data") |> 
  flatten_data() |> 
  group_by(site, Id) |> 
  add_Date_col(group.by = TRUE) |>
  filter_out(any(all(is.na(MEDI_glasses)), all(is.na(MEDI_chest)), all(is.na(MEDI_wrist)))) |> 
  ungroup(Date) |> 
  distinct(Date) |>
  group_by(site) |> 
  count() |> 
  ungroup() |> 
  site_conv_mutate() |> 
  gt() |> 
  cols_label(site = "Site",
             n = "Participant-days") |> 
  grand_summary_rows(
    n, fns = sum ~ sum(.)
  ) |> 
  tab_header("Summary of participant_days")


## Preprocessing step 1

In this step, we will remove non-wear instances that are neither marked as sleep, nor fall into a sleep window (sleepdiary). We will also remove instances ≥ 1.0\*10^5 lx melanopic EDI.

In [ ]:
light <- 
light |> 
  imap(\(x, idx) x |> 
    mutate(across(c(starts_with("MEDI"), starts_with("LIGHT")),
                  \(y) replace_when(y, 
                               wear == "off" & (State.Brown != "sleep" | is.na(State.Brown)) ~ NA,
                               y >= 100000 ~ NA
                               )),
           across(MEDI_glasses:LIGHT_wrist,
                \(z) {
                ifelse(is.na(MEDI_glasses) | is.na(MEDI_chest) | is.na(MEDI_wrist),
                       NA, z)
                }
              )
  )
  )


In [ ]:
light_flat <- 
  structure(light, class = "melidos_data") |> flatten_data() |> group_by(site, Id)


In [ ]:
light_flat |> 
  drop_na(MEDI_glasses) |> 
  add_Date_col() |> 
  distinct(Date) |>
  group_by(site) |> 
  count() |> 
  ungroup() |> 
  site_conv_mutate() |> 
  gt() |> 
  cols_label(site = "Site",
             n = "Participant-days") |> 
  grand_summary_rows(
    n, fns = sum ~ sum(.)
  ) |> 
  tab_header("Summary of remaining participant days after non-wear and out-of-range removal")


## Preprocessing step 2

Here we further process the data: - hours with less than 50% data availability are removed - days with less than 80% data availability (after the previous step) are removed

In [ ]:
light_fin <- 
  light_flat |> 
  cut_Datetime(unit = "1 hour", group_by = TRUE, type = "floor") |> 
  remove_partial_data(MEDI_glasses, threshold.missing = 0.5) |> 
  ungroup(Datetime.rounded) |> 
  select(-Datetime.rounded) |> 
  add_Date_col(group.by = TRUE) |> 
  gap_handler(full.days = TRUE) |> 
  remove_partial_data(MEDI_glasses, threshold.missing = 0.2) |> 
  ungroup(Date)


In [ ]:
light_fin |> 
  distinct(Date) |>
  group_by(site) |> 
  count() |> 
  ungroup() |> 
  site_conv_mutate() |> 
  gt() |> 
  cols_label(site = "Site",
             n = "Participant-days") |> 
  grand_summary_rows(
    n, fns = sum ~ sum(.)
  ) |> 
  tab_header("Summary of remaining participant days after preprocessing")


## Calculate metrics

In [ ]:
datetime_2_numeric <- function(x) {
  x |> 
    mutate(
      across(
        where(is.POSIXct),
        \(x) x |> hms::as_hms() %>% as.numeric()
      )
    )
}


In [ ]:
metrics_data <-
  light_fin |> 
  distinct(site, Id, Datetime, .keep_all = TRUE) |> 
    pivot_longer(MEDI_glasses:LIGHT_wrist,
                 names_sep = "_",
                 names_to = c("metric", "position")) |> 
    pivot_wider(values_from = value, names_from = metric)


In [ ]:
metrics <- 
  metrics_data |> 
  group_by(site, Id, Date, position) |> 
        summarize(
          duration_above_threshold(
            MEDI, Datetime, "above", 10, na.rm = TRUE, as.df = TRUE),
          duration_above_threshold(
            MEDI, Datetime, "above", 250, na.rm = TRUE, as.df = TRUE),
          duration_above_threshold(
            MEDI, Datetime, "above", 1000, na.rm = TRUE, as.df = TRUE),
          period_above_threshold(
            MEDI, Datetime, "above", 10, na.rm = TRUE, as.df = TRUE),
          period_above_threshold(
            MEDI, Datetime, "above", 250, na.rm = TRUE, as.df = TRUE),
          period_above_threshold(
            MEDI, Datetime, "above", 1000, na.rm = TRUE, as.df = TRUE),
          pulses_above_threshold(
            MEDI, Datetime, threshold = 250, na.rm = TRUE, as.df = TRUE
          )|>
            datetime_2_numeric(),
          pulses_above_threshold(
            MEDI, Datetime, threshold = 1000, na.rm = TRUE, as.df = TRUE
          )|>
            datetime_2_numeric(),
          bright_dark_period(
                  MEDI |> log_zero_inflated(),
                  Datetime, "brightest", "10 hours",
                  as.df = TRUE, na.rm = TRUE
                  ) %>%
            datetime_2_numeric(),
          bright_dark_period(
                  MEDI |> log_zero_inflated(),
                  Datetime, "darkest", "10 hours", as.df = TRUE,
                  loop = TRUE, na.rm = TRUE
                  ) %>%
            datetime_2_numeric(),
          timing_above_threshold(
              MEDI, Datetime, "above", 10, as.df = TRUE) |>
            datetime_2_numeric(),
          timing_above_threshold(MEDI, Datetime, "above", 250, as.df = TRUE) |>
            datetime_2_numeric(),
          frequency_crossing_threshold(MEDI, 250, na.rm = TRUE, as.df = TRUE),
          timing_above_threshold(
              MEDI, Datetime, "above", 1000, as.df = TRUE) |>
            datetime_2_numeric(),
          barroso_lighting_metrics(
            MEDI, Datetime, loop = TRUE, na.rm = TRUE, as.df = TRUE
             ),
          centroidLE(MEDI, Datetime, na.rm = TRUE, as.df = TRUE) |>
            datetime_2_numeric(),
          disparity_index(MEDI, TRUE, TRUE),
          midpointCE(MEDI, Datetime, TRUE, TRUE)|>
            datetime_2_numeric(),
          mean_MEDI = mean(MEDI |> log_zero_inflated(), na.rm = TRUE),
          nvRD = nvRD(MEDI, LIGHT, Datetime) |> mean(na.rm = TRUE),
          dose(MEDI, Datetime, na.rm = TRUE, as.df = TRUE),
          MDER = median(MEDI / LIGHT, na.rm = TRUE),
          .groups = "drop",
        ) %>%
        mutate(across(where(is.duration), as.numeric)) %>% 
        pivot_longer(cols = -c(site, Id, Date, position), names_to = "metric")


The first warning was:
ℹ In argument: `nvRD = mean(nvRD(MEDI, LIGHT, Datetime), na.rm = TRUE)`.
ℹ In group 1: `site = "BAUA"`, `Id = "BAUA_S001"`, `Date = 2025-06-11`,
  `position = "chest"`.
Caused by warning in `nvRD()`:
! Light data contains missing values! They are replaced by 0.
ℹ Run `dplyr::last_dplyr_warnings()` to see the 1297 remaining warnings.

In [ ]:
metrics2 <- 
  metrics_data |> 
  group_by(site, Id, position) |> 
  summarize(
    interdaily_stability(MEDI |> log_zero_inflated(), 
                        Datetime, na.rm = TRUE, as.df = TRUE),
    intradaily_variability(MEDI |> log_zero_inflated(), 
                           Datetime, na.rm = TRUE, as.df = TRUE),
          ) |> 
  pivot_longer(cols = -c(site, Id, position), names_to = "metric")


The first warning was:
ℹ In argument: `interdaily_stability(...)`.
ℹ In group 1: `site = "BAUA"`, `Id = "BAUA_S001"`, `position = "chest"`.
Caused by warning in `interdaily_stability()`:
! Data contains some hours with only missing values
These hours contain only missing values: 
[1] "2025-06-12 15:00:00 UTC" "2025-06-12 16:00:00 UTC"
[3] "2025-06-13 15:00:00 UTC" "2025-06-13 16:00:00 UTC"
[5] "2025-06-14 15:00:00 UTC" "2025-06-14 16:00:00 UTC"
ℹ Run `dplyr::last_dplyr_warnings()` to see the 497 remaining warnings.

In [ ]:
metrics <- 
        bind_rows(
          metrics,
          metrics2
          )
rm(metrics2)


## 1-hour-minute values

For the nonlinear explorative analysis, we require 1-hour-values.

In [ ]:
time_data <- 
metrics_data |> 
  group_by(site, Id, Date, position) |> 
  aggregate_Datetime(
    "1 hour",
    type = "floor",
    numeric.handler = \(x) x |> mean(na.rm = TRUE),
    geo.MEDI = MEDI |> log_zero_inflated() |> mean(na.rm = TRUE) |> exp_zero_inflated()
  )|>
  add_Date_col(group.by = TRUE) |> 
  mutate(static = all(MEDI == MEDI[1])) |> 
  filter_out(static) |> 
  select(-static) |> 
  add_Time_col() |> 
  ungroup() |> 
  mutate(Time = as.numeric(Time)/3600 + 0.5,
         across(c(site, Id, sleep, State.Brown, wear, photoperiod.state),
                fct),
         Date = factor(Date),
         Id_date = interaction(Id, Date),
         lzMEDI = log_zero_inflated(MEDI),
         photoperiod = (dusk - dawn) |> as.numeric()
  ) |> 
  group_by(Id, position) |> 
  mutate(AR.start = ifelse(row_number() == 1, TRUE, FALSE)) |> 
  ungroup()


## Export metrics

In [ ]:
save(metrics, file = here("data/prepared_metrics.RData"))
save(time_data, file = here("data/prepared_time_data.RData"))


## Session info

In [ ]:
sessionInfo()


R version 4.5.0 (2025-04-11)
Platform: aarch64-apple-darwin20
Running under: macOS 26.5.2

Matrix products: default
BLAS:   /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.0.dylib 
LAPACK: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRlapack.dylib;  LAPACK version 3.12.1

locale:
[1] en_US.UTF-8/en_US.UTF-8/en_US.UTF-8/C/en_US.UTF-8/en_US.UTF-8

time zone: Europe/Berlin
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices datasets  utils     methods   base     

other attached packages:
 [1] cowplot_1.2.0     rlang_1.2.0       here_1.0.2        gt_1.3.0         
 [5] melidosData_1.0.5 LightLogR_0.10.0  lubridate_1.9.5   forcats_1.0.1    
 [9] stringr_1.6.0     dplyr_1.2.1       purrr_1.2.2       readr_2.2.0      
[13] tidyr_1.3.2       tibble_3.3.1      ggplot2_4.0.2     tidyverse_2.0.0  

loaded via a namespace (and not attached):
 [1] sass_0.4.10        utf8_1.2.6         generics_0.1.4     renv_1.1.4     